# Risk-Neutral Density Extraction — AAPL Options

**Research question:** What does the options market imply about AAPL's 30-day return distribution?

Three methods are compared:
1. **Breeden-Litzenberger (1978)** — model-free second derivative of the call price curve  
2. **SVI (Gatheral 2004)** — 5-parameter parametric smile in total-variance space  
3. **SSVI (Gatheral & Jacquier 2014)** — arbitrage-free surface with butterfly constraint

The implied risk-neutral density (Q-measure) is benchmarked against the empirical historical distribution (P-measure) to quantify the variance risk premium and crash risk premium.

## 1. Imports

In [ ]:
import sys
from pathlib import Path

# Make the local rnd package importable from notebooks/
sys.path.insert(0, str(Path("..") / "src"))

import warnings
import logging
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, gaussian_kde
import yfinance as yf

from rnd import (
    select_expiry, compute_tau, fetch_risk_free_rate,
    fetch_dividend_yield, load_chain,
    preprocess_options, compute_option_ivs,
    svi_total_var, calibrate_svi, rnd_from_svi,
    ssvi_total_var, calibrate_ssvi, rnd_from_ssvi,
    extract_rnd_bl, trapezoidal, clip_normalise,
    compute_tail_metrics, vrp_metrics,
    plotting,
)

# Suppress noisy third-party loggers
for _log in ("yfinance", "peewee"):
    logging.getLogger(_log).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.titlesize": 12})

## 2. Data Loading

In [ ]:
TICKER     = "AAPL"
TARGET_DTE = 30
MIN_OI     = 100
MAX_SPREAD = 0.25

ticker = yf.Ticker(TICKER)

# Current spot price
try:
    S0 = float(ticker.fast_info["last_price"])
except Exception:
    S0 = float(ticker.history(period="1d")["Close"].iloc[-1])

# Select nearest expiry to TARGET_DTE, compute tau
expiry = select_expiry(ticker, TARGET_DTE)
tau    = compute_tau(expiry)

# Risk-free rate (3-month T-bill) and dividend yield
r = fetch_risk_free_rate()
q = fetch_dividend_yield(ticker)

# Forward price
F = S0 * np.exp((r - q) * tau)

# Fetch and liquidity-filter the option chain
calls, puts = load_chain(ticker, expiry, min_oi=MIN_OI, max_spread=MAX_SPREAD)

print(f"{TICKER}  |  S₀={S0:.2f}  F={F:.2f}  r={r:.3%}  q={q:.3%}")
print(f"Expiry: {expiry}  ({tau * 365:.1f} calendar days)")
print(f"Liquid calls: {len(calls)}  |  Liquid puts: {len(puts)}")

## 3. Preprocessing — Put-Call Stitching & Implied Vols

OTM puts (K < F) and OTM calls (K ≥ F) are stitched together for maximum liquidity. Black-Scholes implied vols are computed via Brent's method.

In [ ]:
# Stitch OTM chain and compute implied vols
options = preprocess_options(calls, puts, F)
options = compute_option_ivs(options, S0, r, q, tau)

print(f"OTM options after stitching : {len(preprocess_options(calls, puts, F))}")
print(f"Valid implied vols           : {len(options)}")

# No-arbitrage monotonicity check on OTM call prices
calls_sorted = options[options["type"] == "call"].sort_values("strike")
n_viol = int((np.diff(calls_sorted["mid"].values) > 0).sum())
print(f"Call monotonicity violations : {n_viol}  (0 = arb-free)")

# Plot IV smile
fig = plotting.plot_iv_smile(options, S0, F, title=f"{TICKER} Implied Volatility Smile")
plt.show()

## 4. Breeden-Litzenberger RND (Model-Free)

$$f^Q(K) = e^{r\tau} \frac{\partial^2 C}{\partial K^2}$$

Puts are first converted to call-equivalent prices via put-call parity so the input curve is monotone decreasing — a necessary condition for a non-negative density.

In [ ]:
# Convert puts to call-equivalent via put-call parity
disc = np.exp(-r * tau)
call_equiv = np.where(
    options["type"].values == "put",
    options["mid"].values + F * disc - options["strike"].values * disc,
    options["mid"].values,
)
strikes_bl = options["strike"].values

# Extract Breeden-Litzenberger RND
K_bl, rnd_bl, spline_bl = extract_rnd_bl(strikes_bl, call_equiv, r, tau)

mean_bl = trapezoidal(K_bl, K_bl * rnd_bl)
std_bl  = np.sqrt(trapezoidal(K_bl, (K_bl - mean_bl) ** 2 * rnd_bl))
print(f"B-L RND  |  E^Q[S_T]={mean_bl:.2f}  std={std_bl:.2f}  F={F:.2f}  |  discrepancy={abs(mean_bl - F):.2f}")

fig = plotting.plot_bl_fit(K_bl, rnd_bl, spline_bl, strikes_bl, call_equiv, S0, F, mean_bl)
plt.show()

## 5. SVI Calibration

Total-variance parametrization (Gatheral 2004):

$$w(k) = a + b\left[\rho(k-m) + \sqrt{(k-m)^2 + \sigma^2}\right]$$

Calibration minimises SSE in total-variance space. A lower bound $\sigma \geq 0.05$ prevents degenerate V-shape density artefacts.

In [ ]:
svi_params = calibrate_svi(options["strike"].values, options["iv"].values, F, tau)
a, b, rho_svi, m, sigma_svi = svi_params

atm_iv_svi = float(np.sqrt(svi_total_var(0.0, *svi_params) / tau))
k_mkt = np.log(options["strike"].values / F)
w_mkt = options["iv"].values ** 2 * tau
sse_svi = float(np.sum((svi_total_var(k_mkt, *svi_params) - w_mkt) ** 2))

print(f"SVI  |  a={a:.4f}  b={b:.4f}  ρ={rho_svi:.4f}  m={m:.4f}  σ={sigma_svi:.4f}")
print(f"     |  ATM IV={atm_iv_svi:.2%}  SSE={sse_svi:.2e}")
if sigma_svi < 0.05:
    print("     |  [WARNING] σ < 0.05 — smile nearly V-shaped, density may be degenerate")

# SVI RND
K_svi, rnd_svi = rnd_from_svi(svi_params, F, tau)

# Smile fit + density
k_plot = np.linspace(k_mkt.min() - 0.05, k_mkt.max() + 0.05, 400)
iv_svi_plot = np.sqrt(np.maximum(svi_total_var(k_plot, *svi_params) / tau, 0))

fig = plotting.plot_smile_fit(k_mkt, options["iv"].values, k_plot, iv_svi_plot, title="SVI Smile Fit")
plt.show()

## 6. SSVI Calibration (Arbitrage-Free)

Surface SVI (Gatheral & Jacquier 2014):

$$w(k) = \frac{\theta}{2}\left[1 + \rho\phi k + \sqrt{(\phi k + \rho)^2 + 1 - \rho^2}\right]$$

Note $w(0) = \theta$ by construction. The no-butterfly-arbitrage constraint $\phi \cdot \theta \leq 4$ is enforced as an optimisation bound.

In [ ]:
ssvi_params = calibrate_ssvi(options["strike"].values, options["iv"].values, F, tau)
theta, phi, rho_ssvi = ssvi_params

atm_iv_ssvi = float(np.sqrt(theta / tau))   # w(0) = theta by construction
sse_ssvi = float(np.sum((ssvi_total_var(k_mkt, *ssvi_params) - w_mkt) ** 2))

print(f"SSVI |  θ={theta:.4f}  φ={phi:.4f}  ρ={rho_ssvi:.4f}")
print(f"     |  ATM IV={atm_iv_ssvi:.2%}  SSE={sse_ssvi:.2e}  φ·θ={phi*theta:.4f} ≤ 4 ✓")

# SSVI RND
K_ssvi, rnd_ssvi = rnd_from_ssvi(ssvi_params, F, tau)

# Overlaid smile comparison (SVI + SSVI)
iv_ssvi_plot = np.sqrt(np.maximum(ssvi_total_var(k_plot, *ssvi_params) / tau, 0))

fig = plotting.plot_smile_fit(
    k_mkt, options["iv"].values, k_plot, iv_svi_plot, iv_ssvi=iv_ssvi_plot,
    title="SVI vs SSVI Smile Fit",
)
plt.show()

## 7. Model Comparison

All three densities are clipped to the B-L observed strike range $[K_{\min}, K_{\max}]$ before comparison to avoid spurious visual differences from different support widths.

In [ ]:
K_lo, K_hi = K_bl.min(), K_bl.max()

K_svi_c,  rnd_svi_c  = clip_normalise(K_svi,  rnd_svi,  K_lo, K_hi)
K_ssvi_c, rnd_ssvi_c = clip_normalise(K_ssvi, rnd_ssvi, K_lo, K_hi)

XLIM = (S0 * 0.65, S0 * 1.45)

fig = plotting.plot_model_comparison(
    bl=(K_bl, rnd_bl),
    svi=(K_svi_c, rnd_svi_c),
    ssvi=(K_ssvi_c, rnd_ssvi_c),
    S0=S0,
    xlim=XLIM,
)
plt.show()

## 8. Historical Benchmark — Empirical P-Measure

30-day rolling log-return sums (not $\sqrt{T}$-scaled daily returns) are used to preserve volatility clustering and autocorrelation. A KDE with 0.6× Scott's bandwidth reduces over-smoothing of the fat tails.

In [ ]:
HIST_YEARS = 5
HORIZON    = 30

hist     = ticker.history(start=datetime.now() - timedelta(days=365 * HIST_YEARS))
log_ret  = np.log(hist["Close"] / hist["Close"].shift(1)).dropna()
ret_30d  = log_ret.rolling(HORIZON).sum().dropna()
future_prices = S0 * np.exp(ret_30d.values)

# KDE: 0.6× Scott's rule to avoid over-smoothing fat tails
kde = gaussian_kde(future_prices, bw_method="scott")
kde.set_bandwidth(0.6 * kde.factor)

price_grid = np.linspace(S0 * 0.40, S0 * 2.0, 3000)
rnd_hist_raw = kde(price_grid)
rnd_hist     = rnd_hist_raw / trapezoidal(price_grid, rnd_hist_raw)

hist_ann_vol = float(log_ret.std() * np.sqrt(252))
print(f"Historical (5yr)  |  daily vol={log_ret.std():.4f}  ann={hist_ann_vol:.2%}")
print(f"30-day returns    |  μ={ret_30d.mean():.4f}  σ={ret_30d.std():.4f}  "
      f"skew={ret_30d.skew():.3f}  kurt={ret_30d.kurtosis():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(ret_30d, bins=50, density=True, alpha=0.55, color="steelblue")
x30 = np.linspace(ret_30d.min(), ret_30d.max(), 500)
axes[0].plot(x30, norm.pdf(x30, ret_30d.mean(), ret_30d.std()),
             "r-", lw=2, label="Normal fit")
axes[0].set(xlabel="30-day log-return", ylabel="Density",
            title="Empirical 30-Day Log-Returns")
axes[0].legend()

axes[1].hist(future_prices, bins=60, density=True, alpha=0.40,
             color="steelblue", label="Empirical")
axes[1].plot(price_grid, rnd_hist, "b-", lw=2, label="KDE")
axes[1].axvline(S0, color="red", ls="--", lw=1.2, label=f"S₀={S0:.2f}")
axes[1].set(xlabel="Future price ($)", ylabel="Density",
            title="Historical 30-Day Price Distribution",
            xlim=(S0 * 0.5, S0 * 1.6))
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Risk Metrics & Variance Risk Premium

All tail probabilities are Q-measure (risk-neutral) unless labelled P. VaR and CVaR are expressed as dollar losses from the current spot.

In [ ]:
# Compute tail risk metrics for all four distributions
m_bl   = compute_tail_metrics(K_bl,       rnd_bl,      S0)
m_svi  = compute_tail_metrics(K_svi_c,    rnd_svi_c,   S0)
m_ssvi = compute_tail_metrics(K_ssvi_c,   rnd_ssvi_c,  S0)
m_hist = compute_tail_metrics(price_grid, rnd_hist,     S0)

# Variance risk premium
vrp = vrp_metrics(atm_iv_ssvi, hist_ann_vol)
vrp_add   = vrp["vrp_additive"]
vrp_ratio = vrp["vrp_ratio"]

print(f"Implied vol (SSVI ATM) : {atm_iv_ssvi:.2%}")
print(f"Historical vol (252d)  : {hist_ann_vol:.2%}")
print(f"VRP (vol diff)         : {vrp_add:+.2%}")
print(f"VRP (var ratio)        : {vrp_ratio:.2f}×")

# Summary table
print(f"\n{'Metric':<26} {'B-L':>10} {'SVI':>10} {'SSVI':>10} {'Hist':>10}")
print("-" * 70)
rows = [
    ("Mean ($)",         "mean"),
    ("Std ($)",          "std"),
    ("Skewness",         "skewness"),
    ("Excess kurtosis",  "excess_kurtosis"),
    ("P(−10%)",          "prob_down10"),
    ("P(+10%)",          "prob_up10"),
    ("P(−20% crash)",    "prob_crash20"),
    ("95% VaR ($)",      "var_95"),
    ("95% CVaR ($)",     "cvar_95"),
    ("99% VaR ($)",      "var_99"),
    ("99% CVaR ($)",     "cvar_99"),
]
for label, key in rows:
    vals = [m_bl[key], m_svi[key], m_ssvi[key], m_hist[key]]
    row  = "  ".join(f"{v:10.4f}" if not np.isnan(v) else f"{'n/a':>10}" for v in vals)
    print(f"{label:<26}  {row}")

## 10. Q vs P Comparison & Trading Signals

In [ ]:
# Trading signals derived from Q vs P comparison
vrp_signal  = ("SELL VOL"  if vrp_ratio > 1.20
               else "BUY VOL"  if vrp_ratio < 0.80
               else "NEUTRAL")

crash_ratio = m_ssvi["prob_crash20"] / max(m_hist["prob_crash20"], 1e-6)
tail_signal = ("SELL TAIL" if crash_ratio > 2.0
               else "BUY TAIL"  if crash_ratio < 0.5
               else "NEUTRAL")

skew_diff   = m_ssvi["skewness"] - m_hist["skewness"]
skew_signal = ("RISK REV"  if skew_diff >  0.3
               else "CALL SPRD" if skew_diff < -0.3
               else "NEUTRAL")

print(f"VOL SIGNAL  : {vrp_signal:<10}  (impl={atm_iv_ssvi:.2%}  hist={hist_ann_vol:.2%}  ratio={vrp_ratio:.2f}×)")
print(f"TAIL SIGNAL : {tail_signal:<10}  (Q={m_ssvi['prob_crash20']:.2%}  P={m_hist['prob_crash20']:.2%}  ratio={crash_ratio:.1f}×)")
print(f"SKEW SIGNAL : {skew_signal:<10}  (Δskew={skew_diff:+.3f}  Q={m_ssvi['skewness']:.3f}  P={m_hist['skewness']:.3f})")

fig = plotting.plot_q_vs_p(
    K_ssvi_c, rnd_ssvi_c, price_grid, rnd_hist, S0,
    m_ssvi, m_hist, vrp_add, vrp_ratio,
)
plt.show()

In [ ]:
print("=" * 65)
print(f"RISK-NEUTRAL DENSITY — {TICKER}  |  S₀=${S0:.2f}  |  {expiry}  |  τ={tau*365:.1f}d")
print("=" * 65)
print(f"  r={r:.3%}  q={q:.3%}  F={F:.2f}  |  {len(options)} liquid options")
print()
print("VOLATILITY")
print(f"  Implied vol (SSVI ATM)  : {atm_iv_ssvi:.2%}")
print(f"  Realised vol (252-day)  : {hist_ann_vol:.2%}")
print(f"  VRP                     : {vrp_add:+.2%}  ({vrp_ratio:.2f}×)  →  {vrp_signal}")
print()
print("TAIL RISK (Q-measure)")
print(f"  P^Q(−20% crash)         : {m_ssvi['prob_crash20']:.2%}  vs  P^P={m_hist['prob_crash20']:.2%}  ({crash_ratio:.1f}×)  →  {tail_signal}")
print(f"  95%/99% VaR             : ${m_ssvi['var_95']:.1f} / ${m_ssvi['var_99']:.1f}")
print(f"  95%/99% CVaR            : ${m_ssvi['cvar_95']:.1f} / ${m_ssvi['cvar_99']:.1f}")
print()
print("DISTRIBUTION SHAPE")
print(f"  Implied skewness        : {m_ssvi['skewness']:.4f}  (hist: {m_hist['skewness']:.4f})  →  {skew_signal}")
print(f"  Implied excess kurtosis : {m_ssvi['excess_kurtosis']:.4f}  (hist: {m_hist['excess_kurtosis']:.4f})")
print()
print(f"  {'Method':<24} {'ATM IV':>8} {'Mean':>8} {'Std':>8} {'Skew':>8}")
print("  " + "-" * 54)
for name, m, atm in [
    ("Breeden-Litzenberger",   m_bl,   "—"),
    ("SVI",                    m_svi,  f"{atm_iv_svi:.2%}"),
    ("SSVI",                   m_ssvi, f"{atm_iv_ssvi:.2%}"),
    ("Historical (P)",         m_hist, "—"),
]:
    print(f"  {name:<24} {atm:>8} {m['mean']:>8.2f} {m['std']:>8.2f} {m['skewness']:>8.4f}")
print("=" * 65)